# Variable sensitive analysis

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
import pickle

from thesis.correlation_sociodemographic_covid.util import save_model, summarize_results, \
    tunning_negative_binomial_model

## Loading data

In [2]:
df_variables = pd.read_csv('data/df_without_collinearity_standardized.csv', index_col=0)
df_variables = df_variables.drop(columns=['percentage_population_age_range_60_more', 'percentage_male_population'])

In [3]:
df_political = pd.read_csv('data/df_political_without_missing_points.csv', index_col=0)[['percentual_votes_for_bolsonaro']]

In [4]:
df_y = pd.read_csv('data/df_mortality.csv', index_col=0)

## Variable Sensitivity Analysis

In [7]:
list_columns_y = ['expected_deaths_1s_2020', 'expected_deaths_2020', 'expected_deaths_2021', 'expected_deaths_2022', 'expected_deaths_entire']
list_periods = ['2020_1','2020', '2021', '2022', '2020_2022']

for i in range(5):
    column_y = list_columns_y[i]
    period = list_periods[i]
    print('\n*** Period: ', period)
        
    y = df_y[column_y]

    # Model 11
    print('\*** Model 11')
    print('===>Full model:')
    with open('models/model_11_'+period+'.pkl', 'rb') as file:
        model = pickle.load(file)
    summarize_results(model)

    x = df_variables.copy()
    scaler = StandardScaler()
    percentage_votes_for_bolsonaro_standardized = scaler.fit_transform(df_political)
    x['percentage_votes_for_bolsonaro'] = percentage_votes_for_bolsonaro_standardized[:,0]

    x = x.loc[y>0]
    y = y.loc[y>0]

    for variable in x.columns:
        print('\n*** Removed variable: ',variable)
        x_analysis = x.drop(columns=[variable])
        x_analysis = sm.add_constant(x_analysis)
        model = tunning_negative_binomial_model(x_analysis,y,list_offset=None)
        filename = 'model_11_'+period+'_'+variable
        save_model(model,filename,'models/sensitivity_analysis/variable')
        summarize_results(model)    


*** Period:  2020_1
\*** Model 11
===>Full model:
                    Generalized Linear Model Regression Results                    
Dep. Variable:     expected_deaths_1s_2020   No. Observations:                 3045
Model:                                 GLM   Df Residuals:                     3030
Model Family:             NegativeBinomial   Df Model:                           14
Link Function:                         Log   Scale:                          1.0000
Method:                               IRLS   Log-Likelihood:                -36078.
Date:                     Fri, 20 Jun 2025   Deviance:                       3373.7
Time:                             12:00:18   Pearson chi2:                 4.28e+03
No. Iterations:                         11   Pseudo R-squ. (CS):             0.3161
Covariance Type:                 nonrobust                                         
                                                                 coef    std err          z      P>|z|      [